In [ ]:
!nvidia-smi

Fri Apr 10 05:04:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.3 MB/s eta 0:00:00


In [ ]:
import os
import shutil
from pathlib import Path
from ultralytics import YOLO
import yaml
import torch


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# UPDATE THIS PATH to where you uploaded your dataset in Google Drive
DATASET_ROOT = '/content/drive/MyDrive/kibo-rpc/dataset'

# Working directory
WORK_DIR = '/content/drive/MyDrive/kibo-rpc'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Working directory: {WORK_DIR}")
print(f"Dataset root: {DATASET_ROOT}")

Working directory: /content/drive/MyDrive/kibo-rpc
Dataset root: /content/drive/MyDrive/kibo-rpc/dataset


In [ ]:
# Check if dataset exists
if not os.path.exists(DATASET_ROOT):
    print(f"\n⚠️ ERROR: Dataset not found at {DATASET_ROOT}")
    print("Please update DATASET_ROOT to point to your dataset location in Google Drive")
else:
    print(f"\n✓ Dataset found at {DATASET_ROOT}")

    # List contents
    print("\nDataset contents:")
    for item in os.listdir(DATASET_ROOT):
        item_path = os.path.join(DATASET_ROOT, item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
            subdir_contents = os.listdir(item_path)
            for subitem in subdir_contents[:5]:  # Show first 5 items
                print(f"      - {subitem}")
            if len(subdir_contents) > 5:
                print(f"      ... and {len(subdir_contents) - 5} more")
        else:
            print(f"  📄 {item}")

# Check for required files
required_files = ['data.yaml', 'train', 'valid']
missing_files = [f for f in required_files if not os.path.exists(os.path.join(DATASET_ROOT, f))]
if missing_files:
    print(f"\n⚠️ WARNING: Missing required files/folders: {missing_files}")
else:
    print("\n✓ All required files and folders found")


✓ Dataset found at /content/drive/MyDrive/kibo-rpc/dataset

Dataset contents:
  📄 data.yaml
  📁 train/
      - images
      - labels
      - labels.cache
  📁 test/
      - images
      - labels
  📁 valid/
      - labels
      - images
      - labels.cache

✓ All required files and folders found


In [ ]:
yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')

if not os.path.exists(yaml_path):
    print(f"\n⚠️ ERROR: data.yaml not found at {yaml_path}")
    raise FileNotFoundError(f"data.yaml not found at {yaml_path}")

print(f"\n✓ Loading existing data.yaml from {yaml_path}")

# Read the existing YAML file
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("\nDataset Configuration:")
print(yaml.dump(data_config, sort_keys=False))



✓ Loading existing data.yaml from /content/drive/MyDrive/kibo-rpc/dataset/data.yaml

Dataset Configuration:
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 11
names:
- coin
- compass
- coral
- crystal
- diamond
- emerald
- fossil
- key
- letter
- shell
- treasure_box
roboflow:
  workspace: hankipanki
  project: kibo_rpc_simulation
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/hankipanki/kibo_rpc_simulation/dataset/2



In [ ]:
def count_files_in_dir(dir_path, extensions):
    """Count files with specific extensions in a directory."""
    if not os.path.exists(dir_path):
        return 0
    return len([f for f in os.listdir(dir_path) if any(f.endswith(ext) for ext in extensions)])

print("\nDataset Statistics:")
for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATASET_ROOT, split)
    if os.path.exists(split_path):
        # Check if images and labels are in subdirectories or directly in split folder
        img_path = os.path.join(split_path, 'images') if os.path.exists(os.path.join(split_path, 'images')) else split_path
        lbl_path = os.path.join(split_path, 'labels') if os.path.exists(os.path.join(split_path, 'labels')) else split_path

        img_count = count_files_in_dir(img_path, ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'])
        lbl_count = count_files_in_dir(lbl_path, ['.txt'])

        print(f"  {split.capitalize()}: {img_count} images, {lbl_count} labels")
    else:
        print(f"  {split.capitalize()}: Not found")


Dataset Statistics:
  Train: 420 images, 420 labels
  Val: Not found
  Test: 20 images, 20 labels


In [ ]:
print("\nInitializing YOLOv8 model...")

# Choose model size: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large), yolov8x (extra large)
# Nano is fastest but less accurate, larger models are more accurate but slower
MODEL_SIZE = 'yolov8n.pt'  # Change to 'yolov8s.pt', 'yolov8m.pt', etc. for larger models

model = YOLO(MODEL_SIZE)
print(f"✓ Loaded {MODEL_SIZE}")

# ==================== STEP 9: Training Configuration ====================
EPOCHS = 200  # Number of training epochs (adjust based on results)
IMG_SIZE = 640  # Input image size
BATCH_SIZE = 16  # Batch size (reduce if you get out of memory errors)
PATIENCE = 50  # Early stopping patience

print(f"\nTraining Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Image Size: {IMG_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Patience: {PATIENCE}")


Initializing YOLOv8 model...
✓ Loaded yolov8n.pt

Training Configuration:
  Epochs: 200
  Image Size: 640
  Batch Size: 16
  Patience: 50


In [ ]:
print("\n" + "="*50)
print("Starting Training...")
print("="*50 + "\n")

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    save=True,
    device=0,  # Use GPU (0), set to 'cpu' if no GPU available
    project='runs/detect',
    name='kibo-rpc',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=True,
    val=True,
    plots=True,
    lr0=0.001,
    # Augmentation parameters (can adjust these)
    hsv_h=0.04,  # HSV-Hue augmentation
    hsv_s=0.9,    # HSV-Saturation augmentation
    hsv_v=0.6,    # HSV-Value augmentation
    degrees=20,   # Rotation augmentation
    translate=0.2,  # Translation augmentation
    scale=0.9,    # Scale augmentation
    shear=5.0,    # Shear transformation
    perspective=0.001, # Perspective transformation
    flipud=0.5,   # Vertical flip probability
    fliplr=0.5,   # Horizontal flip probability
    mosaic=1.0,        # Mosaic (100%) - combines 4 images
    mixup=0.15,        # MixUp (15%) - INCREASED - blends 2 images
    copy_paste=0.3,    # Copy-paste (30%) - INCREASED - copies objects between images
    dropout=0.2,       # Dropout - ADDED - prevents overfitting

)

print("\n" + "="*50)
print("Training Completed!")
print("="*50)


Starting Training...

Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/kibo-rpc/dataset/data.yaml, degrees=20, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.04, hsv_s=0.9, hsv_v=0.6, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kibo-rpc, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, 

In [ ]:
print("\nRunning validation...")
# Load YOUR trained model instead
model = YOLO('/content/drive/MyDrive/kibo-rpc/runs/detect/runs/detect/kibo-rpc/weights/best.pt')  # ✅ Has YOUR classes

# Now validate
metrics = model.val(data=yaml_path, split='test')
print("\n📊 Validation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")



Running validation...
Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,793 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 0.1±0.0 MB/s, size: 25.3 KB)
val: Scanning /content/drive/MyDrive/kibo-rpc/dataset/test/labels... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 1.3it/s 14.9s
val: New cache created: /content/drive/MyDrive/kibo-rpc/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.1it/s 1.0s
                   all         20         60      0.827      0.956      0.934      0.706
                  coin          4          5      0.887          1      0.995      0.757
               compass          3          5      0.723        0.8      0.725      0.591
                 coral          4          6      0.931          1      0.995      0.664
               crystal      